
# Preliminary Analysis of Train Passage Acceleration Data
## (Junior) Data Scientist Track – Case Study

This notebook demonstrates a structured approach to:
- Loading binary acceleration data
- Preprocessing signals
- Feature extraction
- Exploratory clustering for train passage comparison


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import rfft, rfftfreq
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os

FS = 2000  # Sampling rate [Hz]


## Helper Functions

In [ ]:

def load_binary_signal(filepath):
    return np.fromfile(filepath, dtype=np.float32)

def preprocess_signal(x, fs):
    x = x - np.mean(x)
    b, a = signal.butter(4, [5/(fs/2), 500/(fs/2)], btype='band')
    return signal.filtfilt(b, a, x)

def extract_time_features(x, fs):
    feats = {}
    feats['rms'] = np.sqrt(np.mean(x**2))
    feats['peak_to_peak'] = np.max(x) - np.min(x)
    feats['variance'] = np.var(x)
    peaks, _ = signal.find_peaks(np.abs(x), height=np.std(x))
    feats['num_peaks'] = len(peaks)
    feats['mean_peak_interval'] = np.mean(np.diff(peaks)/fs) if len(peaks) > 1 else np.nan
    return feats

def extract_frequency_features(x, fs):
    yf = np.abs(rfft(x))
    xf = rfftfreq(len(x), 1/fs)
    return {
        'dominant_frequency': xf[np.argmax(yf)],
        'spectral_centroid': np.sum(xf * yf) / np.sum(yf)
    }


## Load Data

In [ ]:

data_folder = 'data'  # folder containing .dat files
files = [f for f in os.listdir(data_folder) if f.endswith('.dat')]

signals = {f: load_binary_signal(os.path.join(data_folder, f)) for f in files}
print(f"Loaded {len(signals)} signals")


## Visual Inspection

In [ ]:

example_file = list(signals.keys())[0]
x = signals[example_file]
t = np.arange(len(x)) / FS

plt.figure(figsize=(10,4))
plt.plot(t, x)
plt.xlabel("Time [s]")
plt.ylabel("Acceleration [g]")
plt.title("Raw Acceleration Signal")
plt.show()


## Preprocessing

In [ ]:

signals_proc = {k: preprocess_signal(v, FS) for k, v in signals.items()}

plt.figure(figsize=(10,4))
plt.plot(t, signals_proc[example_file])
plt.xlabel("Time [s]")
plt.ylabel("Acceleration [g]")
plt.title("Preprocessed Signal")
plt.show()


## Feature Extraction

In [ ]:

features = []

for name, x in signals_proc.items():
    feats = {}
    feats.update(extract_time_features(x, FS))
    feats.update(extract_frequency_features(x, FS))
    feats['file'] = name
    features.append(feats)

df = pd.DataFrame(features).set_index('file')
df


## Clustering (2 Passages)

In [ ]:

df = df.fillna(df.mean())
X = StandardScaler().fit_transform(df)

kmeans = KMeans(n_clusters=2, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

df



## Conclusion

This notebook demonstrates a full workflow for analyzing vertical acceleration
signals from train passages. While train type and speed cannot be uniquely identified
without ground truth, the methodology reflects a realistic and professional approach.
